<a href="https://colab.research.google.com/github/Rodriamarog/InteligenciaComputacional/blob/main/9_vectorstore.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Vector stores and semantic search



In [1]:
from sentence_transformers import SentenceTransformer

## Part I: Basic vector store implementation

In [6]:
class Document:
    def __init__(self, text: str, metadata: dict[str, str]):
        self.text = text
        self.metadata = metadata


class SearchResult:
    def __init__(self, score: float, document: Document):
        self.score = score
        self.document = document


class VectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        self.model = embedding_model
        self._documents: list[Document] = []
        self._embeddings: np.ndarray | None = None

    def add_documents(self, documents: list[Document]):
        if not documents:
            return

        texts = [doc.text for doc in documents]
        new_embs = self.model.encode(
            texts, batch_size=64, show_progress_bar=True, convert_to_numpy=True
        )
        self._documents.extend(documents)
        self._embeddings = (
            new_embs if self._embeddings is None
            else np.vstack([self._embeddings, new_embs])
        )

    def search(self, query: str, top_k: int = 5) -> list[SearchResult]:
        if not self._documents:
            return []

        q_emb = self.model.encode(query, convert_to_numpy=True)

        q_norm = q_emb / (np.linalg.norm(q_emb) + 1e-10)
        norms = np.linalg.norm(self._embeddings, axis=1, keepdims=True) + 1e-10
        scores = (self._embeddings / norms) @ q_norm

        k = min(top_k, len(self._documents))
        top_idx = np.argpartition(scores, -k)[-k:]
        top_idx = top_idx[np.argsort(scores[top_idx])[::-1]]

        return [
            SearchResult(score=float(scores[i]), document=self._documents[i])
            for i in top_idx
        ]

## Part II: Filtering by metadata

In [7]:
class FilteredVectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        self.model = embedding_model
        self._documents: list[Document] = []
        self._embeddings: np.ndarray | None = None

    def add_documents(self, documents: list[Document]):
        if not documents:
            return

        texts = [doc.text for doc in documents]
        new_embs = self.model.encode(
            texts, batch_size=64, show_progress_bar=True, convert_to_numpy=True
        )
        self._documents.extend(documents)
        self._embeddings = (
            new_embs if self._embeddings is None
            else np.vstack([self._embeddings, new_embs])
        )

    def search(
        self,
        query: str,
        top_k: int = 5,
        metadata_filter: dict[str, str] | None = None,
    ) -> list[SearchResult]:
        if not self._documents:
            return []

        # 1. Obtened indices de documentos que pasan el filtro
        if metadata_filter:
            indices = [
                i for i, doc in enumerate(self._documents)
                if all(doc.metadata.get(k) == v for k, v in metadata_filter.items())
            ]
        else:
            indices = list(range(len(self._documents)))

        if not indices:
            return []

        # 2. Ponerle score solo a los indices filtrados
        sub_matrix = self._embeddings[indices]
        q_emb = self.model.encode(query, convert_to_numpy=True)
        q_norm = q_emb / (np.linalg.norm(q_emb) + 1e-10)
        norms = np.linalg.norm(sub_matrix, axis=1, keepdims=True) + 1e-10
        scores = (sub_matrix / norms) @ q_norm

        k = min(top_k, len(indices))
        top_local = np.argpartition(scores, -k)[-k:]
        top_local = top_local[np.argsort(scores[top_local])[::-1]]

        return [
            SearchResult(score=float(scores[j]), document=self._documents[indices[j]])
            for j in top_local
        ]

In [8]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
import pandas as pd
from sentence_transformers import SentenceTransformer
import numpy as np

# Data classes

class Document:
    def __init__(self, text: str, metadata: dict[str, str]):
        self.text = text
        self.metadata = metadata


class SearchResult:
    def __init__(self, score: float, document: Document):
        self.score = score
        self.document = document


# VectorStore

class VectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        self.model = embedding_model
        self._documents: list[Document] = []
        self._embeddings: np.ndarray | None = None

    def add_documents(self, documents: list[Document]):
        if not documents:
            return

        texts = [doc.text for doc in documents]
        new_embs = self.model.encode(
            texts, batch_size=64, show_progress_bar=True, convert_to_numpy=True
        )
        self._documents.extend(documents)
        self._embeddings = (
            new_embs if self._embeddings is None
            else np.vstack([self._embeddings, new_embs])
        )

    def search(self, query: str, top_k: int = 5) -> list[SearchResult]:
        if not self._documents:
            return []

        q_emb = self.model.encode(query, convert_to_numpy=True)

        q_norm = q_emb / (np.linalg.norm(q_emb) + 1e-10)
        norms = np.linalg.norm(self._embeddings, axis=1, keepdims=True) + 1e-10
        scores = (self._embeddings / norms) @ q_norm

        k = min(top_k, len(self._documents))
        top_idx = np.argpartition(scores, -k)[-k:]
        top_idx = top_idx[np.argsort(scores[top_idx])[::-1]]

        return [
            SearchResult(score=float(scores[i]), document=self._documents[i])
            for i in top_idx
        ]


#  Load dataset

PATH = "/content/drive/MyDrive/InteligenciaComputacional/animal-fun-facts-dataset.csv"

df = pd.read_csv(PATH).dropna(subset=["text"]).reset_index(drop=True)
print(f"Rows loaded: {len(df)}")

documents = [
    Document(
        text=row["text"],
        metadata={
            "animal_name":    str(row.get("animal_name",    "") or ""),
            "source":         str(row.get("source",         "") or ""),
            "media_link":     str(row.get("media_link",     "") or ""),
            "wikipedia_link": str(row.get("wikipedia_link", "") or ""),
        },
    )
    for _, row in df.iterrows()
]

print(f"Documents created: {len(documents)}")


#  Index

model = SentenceTransformer("all-MiniLM-L6-v2")

vs = VectorStore(embedding_model=model)
vs.add_documents(documents)

print(f"Indexed {len(vs._documents)} documents.")


#  Helper

def print_results(query: str, results: list[SearchResult]):
    print("")
    print(f"Query: '{query}'")

    for i, r in enumerate(results, 1):
        print(f"\n[{i}] Score : {r.score:.4f}")
        print(f"    Text  : {r.document.text}")
        print(f"    Animal: {r.document.metadata['animal_name']}")
        print(f"    Source: {r.document.metadata['source']}")
        if r.document.metadata["wikipedia_link"]:
            print(f"    Wiki  : {r.document.metadata['wikipedia_link']}")
        if r.document.metadata["media_link"]:
            print(f"    Media : {r.document.metadata['media_link']}")


#  Queries

print_results("venomous or poisonous animals",
              vs.search("venomous or poisonous animals", top_k=3))

print_results("animals that can fly very fast",
              vs.search("animals that can fly very fast", top_k=3))

print_results("animals that change color or camouflage",
              vs.search("animals that change color or camouflage", top_k=3))

print_results("animals with complex social behavior and communication",
              vs.search("animals with complex social behavior and communication", top_k=3))

print_results("animals that live for a very long time",
              vs.search("animals that live for a very long time", top_k=3))

Rows loaded: 7731
Documents created: 7731


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/121 [00:00<?, ?it/s]

Indexed 7731 documents.

Query: 'venomous or poisonous animals'

[1] Score : 0.7191
    Text  : They are poisonous, not venomous..
This means that they do not inject their toxins into others, like snakes, they instead have to be consumed or licked.
    Animal: poison dart frog
    Source: https://factanimal.com/poison-dart-frog/
    Wiki  : /wiki/Poison_dart_frog
    Media : nan

[2] Score : 0.7178
    Text  : The most venomous fish in the world
    Animal: stonefish
    Source: https://a-z-animals.com/animals/stonefish/
    Wiki  : /wiki/Synanceia
    Media : nan

[3] Score : 0.7108
    Text  : The Most Venomous Snakes On Earth
    Animal: taipan
    Source: https://a-z-animals.com/animals/taipan/
    Wiki  : /wiki/Taipan
    Media : nan

Query: 'animals that can fly very fast'

[1] Score : 0.7108
    Text  : Fastest animal on Earth
    Animal: peregrine falcon
    Source: https://a-z-animals.com/animals/peregrine-falcon/
    Wiki  : /wiki/Peregrine_falcon
    Media : nan

[2] Score :

In [10]:
from datasets import load_dataset

# FilteredVectorStore

class FilteredVectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        self.model = embedding_model
        self._documents: list[Document] = []
        self._embeddings: np.ndarray | None = None

    def add_documents(self, documents: list[Document]):
        if not documents:
            return

        texts = [doc.text for doc in documents]
        new_embs = self.model.encode(
            texts, batch_size=64, show_progress_bar=True, convert_to_numpy=True
        )
        self._documents.extend(documents)
        self._embeddings = (
            new_embs if self._embeddings is None
            else np.vstack([self._embeddings, new_embs])
        )

    def search(
        self,
        query: str,
        top_k: int = 5,
        metadata_filter: dict[str, str] | None = None,
    ) -> list[SearchResult]:
        if not self._documents:
            return []

        if metadata_filter:
            indices = [
                i for i, doc in enumerate(self._documents)
                if all(doc.metadata.get(k) == v for k, v in metadata_filter.items())
            ]
        else:
            indices = list(range(len(self._documents)))

        if not indices:
            return []

        sub_matrix = self._embeddings[indices]
        q_emb = self.model.encode(query, convert_to_numpy=True)
        q_norm = q_emb / (np.linalg.norm(q_emb) + 1e-10)
        norms = np.linalg.norm(sub_matrix, axis=1, keepdims=True) + 1e-10
        scores = (sub_matrix / norms) @ q_norm

        k = min(top_k, len(indices))
        top_local = np.argpartition(scores, -k)[-k:]
        top_local = top_local[np.argsort(scores[top_local])[::-1]]

        return [
            SearchResult(score=float(scores[j]), document=self._documents[indices[j]])
            for j in top_local
        ]


# Load AG News dataset
# 4 categories: World, Sports, Business, Sci/Tech

CATEGORY_MAP = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}

ag = load_dataset("ag_news", split="test")  # 7600 rows

news_documents = [
    Document(
        text=row["text"],
        metadata={
            "category": CATEGORY_MAP[row["label"]],
            "label":    str(row["label"]),
        },
    )
    for row in ag
]

print(f"News documents loaded: {len(news_documents)}")


# Index

fvs = FilteredVectorStore(embedding_model=model)
fvs.add_documents(news_documents)

print(f"Indexed {len(fvs._documents)} documents.")


# Helper

def print_filtered_results(query: str, results: list[SearchResult], fltr: dict):
    print("")
    print(f"Query : '{query}'")
    print(f"Filter: {fltr}")
    for i, r in enumerate(results, 1):
        print(f"\n[{i}] Score   : {r.score:.4f}")
        print(f"    Category: {r.document.metadata['category']}")
        print(f"    Text    : {r.document.text}")


# Queries

print_filtered_results(
    "Olympic Games competition and athletes",
    fvs.search("Olympic Games competition and athletes", top_k=3, metadata_filter={"category": "Sports"}),
    {"category": "Sports"},
)

print_filtered_results(
    "artificial intelligence and machine learning breakthroughs",
    fvs.search("artificial intelligence and machine learning breakthroughs", top_k=3, metadata_filter={"category": "Sci/Tech"}),
    {"category": "Sci/Tech"},
)

print_filtered_results(
    "stock market crash and economic recession",
    fvs.search("stock market crash and economic recession", top_k=3, metadata_filter={"category": "Business"}),
    {"category": "Business"},
)

print_filtered_results(
    "military conflict and war casualties",
    fvs.search("military conflict and war casualties", top_k=3, metadata_filter={"category": "World"}),
    {"category": "World"},
)

print_filtered_results(
    "space exploration and NASA missions",
    fvs.search("space exploration and NASA missions", top_k=3, metadata_filter={"category": "Sci/Tech"}),
    {"category": "Sci/Tech"},
)

News documents loaded: 7600


Batches:   0%|          | 0/119 [00:00<?, ?it/s]

Indexed 7600 documents.

Query : 'Olympic Games competition and athletes'
Filter: {'category': 'Sports'}

[1] Score   : 0.5932
    Category: Sports
    Text    : Olympics-Five sports on shortlist for possible Games inclusion Golf, rugby and squash are on a shortlist of five sports to be assessed for possible inclusion in the 2012 Olympics. The International Olympic Committee is reviewing 

[2] Score   : 0.5748
    Category: Sports
    Text    : Olympics-U.S. Women Show Men How to Win Gold  ATHENS (Reuters) - The U.S. women's basketball team showed  their men how to win gold Saturday as around 70,000 spectators  flocked to the Olympic stadium for a hectic athletics program  on the penultimate night of the Athens Games.

[3] Score   : 0.5718
    Category: Sports
    Text    : Colin Jackson: Hard lessons learnt in the human laboratory Yesterday #39;s Olympics treated us to the two extremes of athletics, the endurance race which tests the body to its limits, and the heavyweight showdown, t